# Create / insert / select / drop on a Jammi mutable companion table

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/f-inverse/jammi-ai/blob/py-v0.49.1/cookbook/notebooks/recipes/mutable_tables.ipynb)

Built from [`cookbook/recipes/mutable_tables/example.py`](https://github.com/f-inverse/jammi-ai/blob/main/cookbook/recipes/mutable_tables/example.py). Run the setup
cell first; every other cell runs top to bottom.

In [ ]:
# Setup: jammi 0.49.1 — the CUDA engine on an sm_80+ GPU (L4, A100, …), the
# CPU engine otherwise — and the cookbook's library and fixtures, from the release's
# tag on GitHub. The chapter runs
# at `small` scale, over the committed fixtures, in minutes. SCALE = "full" runs
# it over the published data and real encoders instead: meant for a GPU, and the
# chapters that fine-tune take hours there.
import os
import subprocess
import sys


def compute_capability() -> float:
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
            capture_output=True, text=True, check=True,
        ).stdout.split()
    except (OSError, subprocess.CalledProcessError):
        return 0.0
    return float(out[0]) if out else 0.0


gpu = compute_capability() >= 8.0
engine = "jammi-ai-native-cu12" if gpu else "jammi-ai-native"
server = "jammi-server-cu12" if gpu else "jammi-server"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jammi-ai==0.49.1", engine + "==0.49.1", "jammi-cookbook @ git+https://github.com/f-inverse/jammi-ai@py-v0.49.1#subdirectory=cookbook/book"], check=True)
SCALE = "small"
os.environ["JAMMI_COOKBOOK_SCALE"] = SCALE
print(f"engine: {engine}   scale: {SCALE}")

Run with `python cookbook/recipes/mutable_tables/example.py`. Exits 0 on
success.

In [ ]:
from __future__ import annotations

import tempfile

import pyarrow as pa

import jammi


def notes_schema() -> pa.Schema:
    return pa.schema(
        [
            pa.field("note_id", pa.int64(), nullable=False),
            pa.field("body", pa.string(), nullable=False),
        ]
    )


def main() -> int:
    with tempfile.TemporaryDirectory() as tmp, jammi.connect(f"file://{tmp}") as db:

        # 1. Create the mutable table. The catalog now resolves
        #    `mutable.public.notes` for SQL DML and reads.
        table_id = db.create_mutable_table(
            "notes",
            schema=notes_schema(),
            primary_key=["note_id"],
        )
        assert table_id == "notes", f"expected 'notes', got {table_id}"

        # 2. Insert three rows through DataFusion's INSERT path.
        db.sql("INSERT INTO mutable.public.notes (note_id, body) VALUES (1, 'one')")
        db.sql("INSERT INTO mutable.public.notes (note_id, body) VALUES (2, 'two')")
        db.sql("INSERT INTO mutable.public.notes (note_id, body) VALUES (3, 'three')")

        # 3. Count + ordered SELECT round-trip the rows we just wrote.
        count = (
            db.sql("SELECT COUNT(*) AS n FROM mutable.public.notes")
            .column("n")
            .to_pylist()[0]
        )
        assert count == 3, f"expected 3 rows, got {count}"

        bodies = (
            db.sql("SELECT body FROM mutable.public.notes ORDER BY note_id")
            .column("body")
            .to_pylist()
        )
        assert bodies == ["one", "two", "three"], f"unexpected rows {bodies}"

        # 4. Drop — after which the federated SQL surface no longer resolves
        #    the table.
        db.drop_mutable_table("notes")
        try:
            db.sql("SELECT COUNT(*) FROM mutable.public.notes")
        except RuntimeError:
            pass
        else:
            raise AssertionError("post-drop SELECT must raise")

        # 5. Idempotent drop — `if_exists=True` does not raise when the
        #    table is already gone.
        db.drop_mutable_table("notes", if_exists=True)

    print("mutable_tables: OK")
    return 0

In [ ]:
assert main() == 0